In [7]:
import torch
import sys

sys.path.insert(0, "..")
from testbed.models import Idefics
import config

device = torch.device("cuda:1")
lmm = Idefics(
    config.idefics_9b_path,
    torch_dtype=torch.bfloat16,
).to(device)

Loading checkpoint shards:   0%|          | 0/19 [00:00<?, ?it/s]

In [2]:
def rotate_half(x):
    """Rotates half the hidden dims of the input."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

# Copied from transformers.models.mixtral.modeling_mixtral.apply_rotary_pos_emb
def apply_rotary_pos_emb(q, k, cos, sin, position_ids, unsqueeze_dim=1):
    """Applies Rotary Position Embedding to the query and key tensors.

    Args:
        q (`torch.Tensor`): The query tensor.
        k (`torch.Tensor`): The key tensor.
        cos (`torch.Tensor`): The cosine part of the rotary embedding.
        sin (`torch.Tensor`): The sine part of the rotary embedding.
        position_ids (`torch.Tensor`):
            The position indices of the tokens corresponding to the query and key tensors. For example, this can be
            used to pass offsetted position ids when working with a KV-cache.
        unsqueeze_dim (`int`, *optional*, defaults to 1):
            The 'unsqueeze_dim' argument specifies the dimension along which to unsqueeze cos[position_ids] and
            sin[position_ids] so that they can be properly broadcasted to the dimensions of q and k. For example, note
            that cos[position_ids] and sin[position_ids] have the shape [batch_size, seq_len, head_dim]. Then, if q and
            k have the shape [batch_size, heads, seq_len, head_dim], then setting unsqueeze_dim=1 makes
            cos[position_ids] and sin[position_ids] broadcastable to the shapes of q and k. Similarly, if q and k have
            the shape [batch_size, seq_len, heads, head_dim], then set unsqueeze_dim=2.
    Returns:
        `tuple(torch.Tensor)` comprising of the query and key tensors rotated using the Rotary Position Embedding.
    """
    cos = cos[position_ids].unsqueeze(unsqueeze_dim)
    sin = sin[position_ids].unsqueeze(unsqueeze_dim)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

def new_forward(
        self,
        hidden_states: torch.Tensor,
        key_value_states= None,
        attention_mask= None,
        position_ids = None,
        past_key_value = None,
        output_attentions: bool = False,
        use_cache: bool = False,
        module_name=None
    ):
        # if key_value_states are provided this layer is used as a cross-attention layer
        is_cross_attention = self.is_cross_attention or key_value_states is not None

        bsz, q_len, _ = hidden_states.size()

        query_states = self.q_proj(hidden_states).view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)
        if not is_cross_attention:
            key_states = self.k_proj(hidden_states).view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)
            value_states = self.v_proj(hidden_states).view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)
        else:
            _, kv_len, _ = key_value_states.size()  # Note that, in this case, `kv_len` == `kv_seq_len`
            key_states = self.k_proj(key_value_states).view(bsz, kv_len, self.num_heads, self.head_dim).transpose(1, 2)
            value_states = (
                self.v_proj(key_value_states).view(bsz, kv_len, self.num_heads, self.head_dim).transpose(1, 2)
            )

        kv_seq_len = key_states.shape[-2]
        if past_key_value is not None:
            kv_seq_len += past_key_value[0].shape[-2]
        if not is_cross_attention:
            cos, sin = self.rotary_emb(value_states, seq_len=max(kv_seq_len, q_len))
            query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin, position_ids)
        # [bsz, nh, t, hd]

        if past_key_value is not None:
            # reuse k, v, self_attention
            key_states = torch.cat([past_key_value[0], key_states], dim=2)
            value_states = torch.cat([past_key_value[1], value_states], dim=2)

        past_key_value = (key_states, value_states) if use_cache else None
       
        if self.qk_layer_norms:
            query_states = self.q_layer_norm(query_states)
            key_states = self.k_layer_norm(key_states)

        if attention_mask is not None:
            if attention_mask.size() != (bsz, 1, q_len, kv_seq_len):
                raise ValueError(
                    f"Attention mask should be of size {(bsz, 1, q_len, kv_seq_len)}, but is {attention_mask.size()}"
                )

        # SDPA with memory-efficient backend is currently (torch==2.1.2) bugged with non-contiguous inputs with custom attn_mask,
        # Reference: https://github.com/pytorch/pytorch/issues/112577.
        if query_states.device.type == "cuda" and attention_mask is not None:
            query_states = query_states.contiguous()
            key_states = key_states.contiguous()
            value_states = value_states.contiguous()

        # We dispatch to SDPA's Flash Attention or Efficient kernels via this `is_causal` if statement instead of an inline conditional assignment
        # in SDPA to support both torch.compile's dynamic shapes and full graph options. An inline conditional prevents dynamic shapes from compiling.
        # The q_len > 1 is necessary to match with AttentionMaskConverter.to_causal_4d that does not create a causal mask in case q_len == 1.
        is_causal = True if self.is_causal and attention_mask is None and q_len > 1 else False

        attn_output = torch.nn.functional.scaled_dot_product_attention(
            query_states,
            key_states,
            value_states,
            attn_mask=attention_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=is_causal,
        )

        if attn_output.size() != (bsz, self.num_heads, q_len, self.head_dim):
            raise ValueError(
                f"`attn_output` should be of size {(bsz, self.num_heads, q_len, self.head_dim)}, but is"
                f" {attn_output.size()}"
            )

        attn_output = attn_output.transpose(1, 2)
        attn_output = attn_output.reshape(bsz, q_len, self.hidden_size)

        attn_output = self.o_proj(attn_output)

        attn_weights = None

        return attn_output, attn_weights, past_key_value

In [3]:
lmm.replace_module_method(r"model\.layers\.\d+\.self\_attn$", "forward", new_forward)

In [4]:
lmm.model

IdeficsForVisionText2Text(
  (model): IdeficsModel(
    (embed_tokens): IdeficsDecoupledEmbedding(
      num_embeddings=32000, num_additional_embeddings=2, embedding_dim=4096, partially_freeze=False
      (additional_embedding): Embedding(2, 4096)
    )
    (vision_model): IdeficsVisionTransformer(
      (embeddings): IdeficsVisionEmbeddings(
        (patch_embedding): Conv2d(3, 1280, kernel_size=(14, 14), stride=(14, 14), bias=False)
        (position_embedding): Embedding(257, 1280)
      )
      (pre_layrnorm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
      (encoder): IdeficsVisionEncoder(
        (layers): ModuleList(
          (0-31): 32 x IdeficsVisionEncoderLayer(
            (self_attn): IdeficsVisionAttention(
              (k_proj): Linear(in_features=1280, out_features=1280, bias=True)
              (v_proj): Linear(in_features=1280, out_features=1280, bias=True)
              (q_proj): Linear(in_features=1280, out_features=1280, bias=True)
              (out_p

In [5]:
import pytorch_lightning as pl
import datasets
from torch.utils.data import (
    DistributedSampler,
    BatchSampler,
    SequentialSampler,
    RandomSampler,
)
import os
import sys
from pathlib import Path

from testbed.data import prepare_caption_input, prepare_dataloader, prepare_vqa_input
import config
import exp_settings as setting


class ICVDataModule(pl.LightningDataModule):

    def __init__(self, lmm) -> None:
        super().__init__()
        self.lmm = lmm
        if setting.dataset == "vqav2":
            self.dataset = datasets.load_dataset(
                os.path.join(config.testbed_dir, "data", "vqav2"),
                split="train",
                data_dir=config.vqav2_dir,
                images_dir=config.coco_dir,
                trust_remote_code=True,
            )
        elif setting.dataset == "ok_vqa":
            self.dataset = datasets.load_dataset(
                os.path.join(config.testbed_dir, "data", "ok_vqa"),
                split="train",
                data_dir=config.ok_vqa_dir,
                images_dir=config.coco_dir,
                trust_remote_code=True,
            )
        elif setting.dataset == "coco_cap":
            self.dataset = datasets.load_dataset(
                os.path.join(config.testbed_dir, "data", "coco"),
                split="train",
                data_dir=config.karpathy_coco_caption_dir,
                images_dir=config.coco_dir,
                trust_remote_code=True,
            )
        self.query_set = self.dataset.select(
            range(setting.num_query_samples)
        )

    def collate_fn(self, batch):
        """
        Split batch into full context, in-context examples, query and answer, and process them into model inputs.
        """
        if setting.dataset in ["vqav2", "ok_vqa"]:
            context, images = prepare_vqa_input(
                batch, instruction=setting.vqa_instruction
            )
            # we use the first answer as grounding truth
            answers = [item[-1]["answers"][0]["answer"] for item in batch]
        elif setting.dataset == "caption":
            context, images = prepare_caption_input(
                batch, instruction=setting.caption_instruction
            )
            answers = [item[-1]["sentences_raw"][0] for item in batch]

        # the last two items (take vqa as an example):
        # [
        #   { "role" : "question"
        #     "content" :  ... },
        #   { "role" : "short answer" }
        # ]
        ice_texts = self.lmm.apply_prompt_template([ctx[:-2] for ctx in context])
        query_texts = self.lmm.apply_prompt_template([ctx[-2:] for ctx in context])

        return {
            "ice_texts": ice_texts,
            "query_texts": query_texts,
            "answers": answers,
            "images": images,
        }

    def train_dataloader(self):
        if self.trainer and self.trainer.world_size > 1:
            samplers = [
                BatchSampler(
                    DistributedSampler(self.dataset, shuffle=False),
                    batch_size=setting.num_shot,
                    drop_last=True,
                ),
                DistributedSampler(self.query_set, shuffle=False),
            ]
        else:
            samplers = [
                BatchSampler(
                    SequentialSampler(self.dataset), batch_size=1, drop_last=True
                ),
                SequentialSampler(self.query_set),
            ]

        return prepare_dataloader(
            [self.dataset, self.query_set],
            2,
            num_per_dataset=[1, 1],
            collate_fn=self.collate_fn,
            samplers=samplers,
            # num_workers=1,
            pin_memory=True,
            shuffle=False,
        )

datamodule = ICVDataModule(lmm)
dataloader = datamodule.train_dataloader()

In [6]:
import pytorch_lightning as pl
import torch
import torch.nn.functional as F
import torch.optim as optim
from deepspeed.ops.adam import DeepSpeedCPUAdam
from transformers import get_cosine_schedule_with_warmup

import exp_settings as setting
from testbed.models.model_base import HookType


class ICVModel(pl.LightningModule):
    def __init__(self, lmm, icv_encoder: torch.nn.Module) -> None:
        super().__init__()
        self.lmm = lmm
        self.lmm.requires_grad_(False)
        self.icv_encoder = icv_encoder

    def generate_label_mask(self, inputs, num_separator, keep_bos=False):
        """
        Generates label mask which masks tokens before num_separator pad_tokens from given inputs.
        """
        input_ids = inputs["input_ids"]
        batch_size, seq_len = input_ids.shape
        pad_mask = input_ids == self.lmm.processor.tokenizer.pad_token_id
        non_pad_mask = ~pad_mask
        label_mask = torch.zeros_like(input_ids, dtype=torch.bool)
        if self.lmm.processor.tokenizer.padding_side == "left":
            eos_position = non_pad_mask.int().argmax(dim=1)

        for i in range(batch_size):
            seq_pad_positions = pad_mask[i].nonzero(as_tuple=False).squeeze(-1)
            num_pads = len(seq_pad_positions)
            if num_pads < num_separator:
                raise ValueError(
                    f"Sequence {i} has fewer pad tokens ({num_pads}) than num_separator ({num_separator})"
                )

            if self.lmm.processor.tokenizer.padding_side == "left":
                seq_pad_positions = seq_pad_positions[
                    seq_pad_positions > eos_position[i]
                ]

            sep_position = seq_pad_positions[num_separator - 1].item()
            label_mask[i, sep_position + 1 :] = True

        return label_mask & non_pad_mask

    def kl_with_last_logits(self, ice_texts, query_texts, answers, images):
        pad_token, pad_token_id, eos_token = (
            self.lmm.processor.tokenizer.pad_token,
            self.lmm.processor.tokenizer.pad_token_id,
            self.lmm.processor.tokenizer.eos_token,
        )

        hooks = self.icv_encoder.register_hook_for(self.lmm)

        # step 1. ICV + query + [PAD] + answer [EOS] forward process
        query_answer = [
            query + pad_token + answer + eos_token
            for query, answer in zip(query_texts, answers)
        ]
        query_images = [img[-setting.num_image_in_query :] for img in images]
        query_inputs = self.lmm.process_input(query_answer, query_images).to(
            self.device
        )
        query_inputs["attention_mask"] = query_inputs["input_ids"] != pad_token_id
        query_outputs = self.lmm.model(
            **query_inputs,
            labels=query_inputs["input_ids"],
        )
        icv_logits = query_outputs["logits"]
        for hook in hooks:
            hook.remove()

        # step 2. ICE + query + [PAD] answer [EOS] forward process
        full_text = [
            ice + query + pad_token + answer + eos_token
            for ice, query, answer in zip(ice_texts, query_texts, answers)
        ]
        inputs = self.lmm.process_input(full_text, images).to(self.device)
        inputs["attention_mask"] = inputs["input_ids"] != pad_token_id
        with torch.no_grad():
            ice_logits = self.lmm.model(**inputs)["logits"]

        # step 3. extract answer logits & calculate kl divergency
        kl_loss = F.kl_div(
            icv_logits[self.generate_label_mask(query_inputs, 1)].log_softmax(dim=-1),
            ice_logits[self.generate_label_mask(inputs, 1)].softmax(dim=-1),
            reduction="batchmean",
            log_target=False,
        )
        ce_loss = query_outputs["loss"]
        total_loss = kl_loss + setting.ce_loss_weight * ce_loss

        return {
            "kl_loss": kl_loss,
            "ce_loss": ce_loss,
            "loss": total_loss,
        }

    def kl_each_layer(self, ice_texts, query_texts, answers, images):
        pad_token, pad_token_id, bos_token_id, eos_token = (
            self.lmm.processor.tokenizer.pad_token,
            self.lmm.processor.tokenizer.pad_token_id,
            self.lmm.processor.tokenizer.bos_token_id,
            self.lmm.processor.tokenizer.eos_token,
        )

        hooks, record_hooks = self.icv_encoder.register_hook_for(self.lmm)

        # step 1. ICV + query + [PAD] + answer [EOS] forward process
        query_answer = [
            query + pad_token + answer + eos_token
            for query, answer in zip(query_texts, answers)
        ]
        query_images = [img[-setting.num_image_in_query :] for img in images]
        query_inputs = self.lmm.process_input(query_answer, query_images).to(
            self.device
        )
        query_inputs["attention_mask"] = query_inputs["input_ids"] != pad_token_id
        self.lmm.model(**query_inputs)
        query_label_mask = query_inputs["attention_mask"] & (
            query_inputs["input_ids"] != bos_token_id
        )
        icv_hidden_states = torch.cat(
            [hs[query_label_mask] for hs in self.icv_encoder.hidden_states]
        )
        for hook in hooks:
            hook.remove()

        # step 2. ICE [PAD] query [PAD] answer [EOS] forward process
        full_text = [
            ice + pad_token + query + pad_token + answer + eos_token
            for ice, query, answer in zip(ice_texts, query_texts, answers)
        ]
        inputs = self.lmm.process_input(full_text, images).to(self.device)
        inputs["attention_mask"] = inputs["input_ids"] != pad_token_id
        self.lmm.model(**inputs)["logits"]
        # extract query [PAD] answer [EOS] from hidden_states
        query_label_mask = self.generate_label_mask(inputs, 1)
        ice_hidden_states = torch.cat(
            [hs[query_label_mask] for hs in self.icv_encoder.hidden_states]
        )
        for hook in record_hooks:
            hook.remove()

        # step 3. extract answer logits & calculate kl divergency
        layer_kl_loss = F.kl_div(
            icv_hidden_states.log_softmax(dim=-1),
            ice_hidden_states.softmax(dim=-1),
            reduction="batchmean",
            log_target=False,
        )

        return {
            "loss": layer_kl_loss,
        }

    def forward(self, ice_texts, query_texts, answers, images):
        if (
            not hasattr(self.icv_encoder, "hidden_states")
            or self.icv_encoder.hidden_states is None
        ):
            self.kl_with_last_logits(ice_texts, query_texts, answers, images)
        return self.kl_each_layer(ice_texts, query_texts, answers, images)

    def training_step(self, batch, batch_idx):
        loss_dict = self(**batch)

        self.log_dict(loss_dict, sync_dist=True, prog_bar=True)

        for i, alpha in enumerate(self.icv_encoder.alpha):
            self.log(f"alpha/alpha-{i}", alpha.item())

        return loss_dict["loss"]

    def configure_optimizers(self):
        param_dict = {
            pn: p for pn, p in self.icv_encoder.named_parameters() if p.requires_grad
        }

        alpha_params = [p for n, p in param_dict.items() if "alpha" in n]
        non_alpha_params = [p for n, p in param_dict.items() if not "alpha" in n]

        optim_groups = [
            {"params": non_alpha_params, "lr": setting.icv_lr},
            {"params": alpha_params, "lr": setting.alpha_lr},
        ]

        if "deepspeed" in setting.strategy:
            optimizer = DeepSpeedCPUAdam(
                optim_groups,
                weight_decay=setting.weight_decay,
            )
        else:
            optimizer = optim.Adam(
                optim_groups,
                weight_decay=setting.weight_decay,
            )

        step_batches = self.trainer.estimated_stepping_batches
        warmup_steps = setting.warmup_step
        if isinstance(warmup_steps, float):
            warm_steps = warmup_steps * step_batches
        elif isinstance(warmup_steps, int):
            warm_steps = warmup_steps
        else:
            raise ValueError(
                f"the warm_steps should be int or float, but got {type(warmup_steps)}"
            )
        scheduler = get_cosine_schedule_with_warmup(
            optimizer, num_warmup_steps=warm_steps, num_training_steps=step_batches
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "interval": "step"},
        }

    def on_save_checkpoint(self, checkpoint):
        checkpoint["state_dict"] = {
            k: v
            for k, v in checkpoint["state_dict"].items()
            if not k.startswith("model")
        }
        return checkpoint

[2024-09-20 19:35:27,337] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/jyc/miniconda3/envs/icl/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/jyc/miniconda3/envs/icl/compiler_compat/ld: warning: libstdc++.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/jyc/miniconda3/envs/icl/compiler_compat/ld: warning: libm.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/jyc/miniconda3/envs/icl/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `std::runtime_error::~runtime_error()@GLIBCXX_3.4'
/home/jyc/miniconda3/envs/icl/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `__gxx_personality_v0@CXXABI_1.3'
/home/jyc/miniconda3/envs/icl/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `std::ostream::tellp()@GLIBCXX_3.4'
/home/jyc/miniconda3/envs/icl/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: und

In [7]:
from testbed.models.model_base import HookType
import torch.nn as nn
import re

class GlobalICVEncoder(nn.Module):
    def __init__(
        self, lmm_hidden_dim, lmm_layers, alpha_init_value=0.1, record_hidden_states=False, **kwargs
    ) -> None:
        super().__init__()

        self.alpha = torch.nn.Parameter(
            torch.full((lmm_layers,), fill_value=alpha_init_value)
        )
        self.icv = torch.nn.Parameter(
            torch.empty(lmm_layers, lmm_hidden_dim).normal_(mean=0.0, std=0.01)
        )
        self.hidden_states = [[] for _ in range(lmm_layers)] if record_hidden_states else None 
        

    def register_hook_for(self, lmm, **model_inputs):
        hooks = lmm.register_forward_hook(HookType.TEXT_MODEL_LAYER, self.hook)
        if self.hidden_states:
            return hooks, lmm.register_forward_hook(HookType.TEXT_MODEL_LAYER, self.record_hook)
        return hooks

    def record_hook(self, m, inputs, outputs, module_name, **kwargs):
        layer_idx = int(re.findall(r"\d+", module_name)[0])
        if not isinstance(outputs, tuple):
            outputs = (outputs,)
        hidden_states, *rest = outputs
        self.hidden_states[layer_idx] = hidden_states

    def hook(self, m, inputs, outputs, module_name, **kwargs):
        layer_idx = int(re.findall(r"\d+", module_name)[0])
        if not isinstance(outputs, tuple):
            outputs = (outputs,)
        hidden_states, *rest = outputs
        shift = self.alpha[layer_idx] * self.icv[layer_idx]
        shifted_states = hidden_states + shift[None, None, :]
        normalized_states = (
            shifted_states
            / shifted_states.norm(dim=-1, keepdim=True)
            * hidden_states.norm(dim=-1, keepdim=True)
        )
        return normalized_states, *rest


torch.manual_seed(426)
icv_encoder = GlobalICVEncoder(4096, 32, record_hidden_states=True).to(device, torch.float16)
model = ICVModel(lmm, icv_encoder).to(device, torch.float16)

In [8]:
inputs = next(iter(dataloader))
model(**inputs)

nice try, model.layers.0.self_attn
nice try, model.layers.1.self_attn
nice try, model.layers.2.self_attn
nice try, model.layers.3.self_attn
nice try, model.layers.4.self_attn
nice try, model.layers.5.self_attn
nice try, model.layers.6.self_attn
nice try, model.layers.7.self_attn
nice try, model.layers.8.self_attn
nice try, model.layers.9.self_attn
nice try, model.layers.10.self_attn
nice try, model.layers.11.self_attn
nice try, model.layers.12.self_attn
nice try, model.layers.13.self_attn
nice try, model.layers.14.self_attn
nice try, model.layers.15.self_attn
nice try, model.layers.16.self_attn
nice try, model.layers.17.self_attn
nice try, model.layers.18.self_attn
nice try, model.layers.19.self_attn
nice try, model.layers.20.self_attn
nice try, model.layers.21.self_attn
nice try, model.layers.22.self_attn
nice try, model.layers.23.self_attn
nice try, model.layers.24.self_attn
nice try, model.layers.25.self_attn
nice try, model.layers.26.self_attn
nice try, model.layers.27.self_attn
ni

{'loss': tensor(0.6685, device='cuda:1', dtype=torch.float16, grad_fn=<DivBackward0>)}

In [9]:
output = model(**inputs)


nice try, model.layers.0.self_attn
nice try, model.layers.1.self_attn
nice try, model.layers.2.self_attn
nice try, model.layers.3.self_attn
nice try, model.layers.4.self_attn
nice try, model.layers.5.self_attn
nice try, model.layers.6.self_attn
nice try, model.layers.7.self_attn
nice try, model.layers.8.self_attn
nice try, model.layers.9.self_attn
nice try, model.layers.10.self_attn
nice try, model.layers.11.self_attn
nice try, model.layers.12.self_attn
nice try, model.layers.13.self_attn
nice try, model.layers.14.self_attn
nice try, model.layers.15.self_attn
nice try, model.layers.16.self_attn
nice try, model.layers.17.self_attn
nice try, model.layers.18.self_attn
nice try, model.layers.19.self_attn
nice try, model.layers.20.self_attn
nice try, model.layers.21.self_attn
nice try, model.layers.22.self_attn
nice try, model.layers.23.self_attn
nice try, model.layers.24.self_attn
nice try, model.layers.25.self_attn
nice try, model.layers.26.self_attn
nice try, model.layers.27.self_attn
ni

In [25]:
from jinja2 import Environment

# 示例数据
messages = [
    {
        "role": "instruction",
        "content": "Provide an answer to the question. Use the image to answer.",
    },
    {
        "role": "question",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "What do we see in this image?"},
        ],
    },
    {
        "role": "short answer",
        "content": [
            {
                "type": "text",
                "text": "In this image, we can see the city of New York, and more specifically the Statue of Liberty.",
            },
        ],
    },
    {
        "role": "question",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "And how about this image?"},
        ],
    },
    {
        "role": "short answer"
    }
]

# 模板字符串
template_string = """{% if messages[0]['role'] == 'instruction' -%}
Instruction: {{ messages[0]['content'] }}
{%- set messages = messages[1:] %}
{%- endif %}
{%- for message in messages %}
{%- if 'content' in message and message['content'] and message['content'][0]['type'] == 'image' %}
\n<image>
{%- endif %} {{ message['role'].capitalize() }}: {%- if 'content' in message %}
{%- for line in message['content'] %}
{%- if line['type'] == 'text' %} {{ line['text'] }}{%- endif %}
{%- endfor %}{%- endif %}
{%- if not loop.last %}\n{%- endif %}
{%- endfor %}"""
lmm.apply_prompt_template(messages, template_string)

'Instruction: Provide an answer to the question. Use the image to answer.\n<image> Question: What do we see in this image? Short answer: In this image, we can see the city of New York, and more specifically the Statue of Liberty.\n<image> Question: And how about this image? Short answer:'

: 